# Kişi 2 — Colab

Çalışma zamanı: **T4 GPU**. Menü: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4.

`veri_isleme.py` çalıştırma. `data/splits.csv` kilitlidir.

Oturum kapanmadan `results` klasörünü indir.

In [ ]:
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'GPU kapalı. Çalışma zamanını T4 yapıp yeniden başlat.'

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path
uploaded = files.upload()
name = next(iter(uploaded))
assert name.endswith('.zip')
zipfile.ZipFile(name).extractall('/content')
%cd /content/Final-project
!ls src configs data

In [ ]:
!pip -q install -r requirements.txt datasets

In [ ]:
from pathlib import Path
import json
import pandas as pd
from datasets import load_dataset

splits = pd.read_csv('data/splits.csv')
assert len(splits) == 1457
assert (splits.kume == 'test').sum() == 292
assert splits.file_name.nunique() == 1457

out = Path('data/images')
out.mkdir(parents=True, exist_ok=True)
need = set(splits.file_name) - {p.name for p in out.glob('*.jpg')}
rows = load_dataset('Layered-Labs/ACNE04')['train']
boxes = {}
for row in rows:
    name = row['file_name']
    if name in need:
        row['image'].save(out / name)
    boxes[name] = {
        'file_name': name,
        'width': row['image'].width,
        'height': row['image'].height,
        'xmin': list(row['lesion_boxes']['xmin']),
        'ymin': list(row['lesion_boxes']['ymin']),
        'xmax': list(row['lesion_boxes']['xmax']),
        'ymax': list(row['lesion_boxes']['ymax']),
    }
with open('data/boxes.jsonl', 'w', encoding='utf-8') as handle:
    for name in splits.file_name:
        handle.write(json.dumps(boxes[name]) + '\n')
print('jpg', len(list(out.glob('*.jpg'))))
assert len(list(out.glob('*.jpg'))) == 1457

In [ ]:
!python src/yolo_hazirla.py

In [ ]:
!python src/train.py --model efficientnet_b0 --aug on --split data/splits.csv

In [ ]:
!python src/evaluate.py --model efficientnet_b0 --weights results/efficientnet_b0_best.pt --split test --write-csv
!python src/kazanan.py

In [ ]:
!yolo detect train model=yolov8n.pt data=data/yolo/data.yaml imgsz=640 batch=16 epochs=50 seed=42 name=acne04 project=runs/detect
!cp runs/detect/acne04/weights/best.pt results/yolov8n_best.pt

In [ ]:
!python src/sayimdan_siddet.py --weights results/yolov8n_best.pt
!cat results/results.csv results/sifir_kutu.txt

In [ ]:
!zip -r /content/kisi2_sonuc.zip results/efficientnet_b0_best.pt results/efficientnet_b0_egri.png results/efficientnet_b0_val.json results/cm_efficientnet_b0.png results/yolov8n_best.pt results/results.csv results/sifir_kutu.txt results/kazanan.txt
from google.colab import files
files.download('/content/kisi2_sonuc.zip')